# 3주차 금융 개인·팀 프로젝트 · 학생용 빈칸

## 프로젝트의 업무 목표

2주차에 만든 **가맹점 240곳의 30일 요약표**를 실제 서비스가 사용할 수 있는 형태로 연결합니다.

1. 저장된 전처리기를 다시 사용합니다.
2. SQLite에 저장하고 SQL로 확인합니다.
3. EDA와 PCA로 데이터의 특징을 확인합니다.
4. FastAPI가 데이터를 제공하고 Streamlit이 화면에 보여 주도록 준비합니다.

이 노트북의 한 행은 **가맹점 한 개**를 뜻합니다. 시작 데이터는 240행×26열의 DataFrame입니다.

> manual_review_required는 수업용 운영 라벨이며 실제 신용·제재 결정을 대신하지 않습니다.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sqlalchemy import create_engine

def find_project_dir():
    """노트북을 어느 위치에서 열어도 이 과제 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base,
            base / "course_materials" / "assignments" / "week3" / "option2_finance",
            base / "assignments" / "week3" / "option2_finance",
        ]
        for candidate in candidates:
            if (candidate / "data" / "finance_merchant_summary.csv").exists():
                return candidate
    raise FileNotFoundError("finance_merchant_summary.csv를 찾지 못했습니다. 과제 폴더가 포함된 프로젝트를 열어 주세요.")

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATABASE_DIR = PROJECT_DIR / "database"
FIGURE_DIR = PROJECT_DIR / "figures"
for folder in [OUTPUT_DIR, DATABASE_DIR, FIGURE_DIR]:
    folder.mkdir(exist_ok=True)

ID_COLUMN = "merchant_id"
SEGMENT_COLUMN = "industry"
TARGET_COLUMN = "manual_review_required"
NUMERIC_FEATURES = ['monthly_sales_10k_krw', 'refund_rate', 'chargeback_rate', 'transaction_count', 'active_days', 'total_amount_krw', 'avg_amount_krw', 'approval_rate', 'refund_rate_30d', 'avg_processing_ms', 'risk_signal_total', 'cross_border_count']
CATEGORICAL_FEATURES = ['industry', 'region', 'kyc_status', 'risk_tier']
INPUT_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("프로젝트 폴더:", PROJECT_DIR)

## Q1. 2주차 결과가 프로젝트의 시작 데이터로 준비되었나요?

`data/finance_merchant_summary.csv`은 1주차 기준표와 2주차 30일 로그 요약을 결합한 기준본입니다.

학생이 할 일:

- `pd.read_csv`로 파일을 DataFrame에 읽습니다.
- 행과 열의 수가 `(240, 26)`인지 확인합니다.
- `merchant_id`가 중복되지 않는지 확인합니다.

In [ ]:
source_path = DATA_DIR / "finance_merchant_summary.csv"
project_data = pd.FILL_ME(source_path)

print(project_data.shape)
print(project_data[["merchant_id", "merchant_name", "industry"]].head(3))
assert project_data.shape == (240, 26)
assert project_data["merchant_id"].FILL_ME

## Q2. 2주차 전처리기와 PCA로 240개 대상을 두 숫자로 표현합니다

`joblib.load`는 2주차에 저장한 전처리기 객체를 다시 불러옵니다. 이 전처리기는 어떤 숫자 열을 채우고 표준화할지, 어떤 문자 열을 원핫 인코딩할지 기억하고 있습니다.

여기서는 전처리기를 새로 학습하지 않습니다. 이미 저장된 기준에 `transform`만 적용합니다. 그 결과에 PCA를 적용해 `pca_1`, `pca_2`라는 좌표 두 개를 만듭니다.

In [ ]:
artifact_path = ARTIFACT_DIR / "finance_preprocessor.joblib"
preprocessor = joblib.load(artifact_path)

model_input = project_data[INPUT_FEATURES].copy()
processed_values = preprocessor.FILL_ME(model_input)

pca = PCA(n_components=2, random_state=42)
coordinates = pca.FILL_ME(processed_values)
project_data["pca_1"] = coordinates[:, 0]
project_data["pca_2"] = coordinates[:, 1]

print("전처리 결과:", processed_values.shape)
print("PCA 좌표:", coordinates.shape)
print("두 축 설명력:", round(pca.explained_variance_ratio_.sum() * 100, 1), "%")
assert coordinates.shape == (240, 2)

## Q3. DataFrame을 SQLite에 저장한 뒤 정말 240행인지 다시 확인합니다

메모리의 DataFrame은 Python을 종료하면 사라집니다. SQLite에 저장하면 앱을 다시 실행해도 같은 데이터를 조회할 수 있습니다.

`if_exists="replace"`는 같은 이름의 실습 테이블이 있으면 이번 결과로 바꿉니다. 실제 누적 수집 업무라면 `append`와 중복 방지 규칙을 함께 설계해야 합니다.

In [ ]:
db_path = DATABASE_DIR / "finance_project.db"
engine = create_engine(f"sqlite:///{db_path}")

project_data.FILL_ME(
    "merchant_dashboard", engine,
    if_exists="FILL_ME", index=False
)

row_check = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM merchant_dashboard",
    engine
)
print(row_check)
assert row_check.loc[0, "row_count"] == 240

## Q4. 업종별로 무엇이 다른지 EDA 표와 PCA 그림을 만듭니다

EDA의 목적은 그림을 많이 만드는 것이 아닙니다. **어느 집단에서 어떤 숫자가 다른지** 확인하고 다음 질문을 고르는 것입니다.

이 과제에서는 `industry`별 대상 수, `risk_signal_total` 평균, `approval_rate` 평균, `manual_review_required` 개수를 한 표에서 비교합니다.

In [ ]:
segment_summary = project_data.FILL_ME("industry", as_index=False).FILL_ME(
    object_count=("merchant_id", "count"),
    avg_metric=("risk_signal_total", "mean"),
    avg_rate=("approval_rate", "mean"),
    target_count=("manual_review_required", "sum"),
)
display(segment_summary)

figure, axes = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=project_data,
    x="pca_1", y="pca_2",
    hue="manual_review_required",
    ax=axes
)
axes.set_title("금융 PCA 2차원 위치")
figure.tight_layout()
figure_path = FIGURE_DIR / "finance_pca.png"
figure.savefig(figure_path, bbox_inches="tight")
plt.show()
print("그림 저장:", figure_path)

## Q5. API와 Streamlit이 함께 읽을 최종 CSV를 저장합니다

`status_label`은 True/False를 화면에서 읽기 쉬운 말로 바꾼 열입니다. 원래 정답 열 `manual_review_required`도 남겨 두므로 정보가 사라지지 않습니다.

저장 파일은 이후 `api_app_student.py`와 `dashboard_student.py`가 함께 읽습니다.

In [ ]:
project_data["status_label"] = np.where(
    project_data["manual_review_required"],
    "확인 필요", "일반"
)
dashboard_columns = [
    "merchant_id", "merchant_name", "industry",
    "risk_signal_total", "approval_rate",
    "manual_review_required", "status_label", "pca_1", "pca_2"
]
dashboard_data = project_data[dashboard_columns].copy()
dashboard_path = OUTPUT_DIR / "finance_dashboard_ready.csv"
dashboard_data.FILL_ME(dashboard_path, index=False, encoding="utf-8-sig")

print(dashboard_data.shape)
print("저장 파일:", dashboard_path)
assert dashboard_data.shape == (240, 9)
assert dashboard_path.exists()

## Q6. 숫자 두 개를 근거로 한 문장 인사이트를 작성합니다

다음 세 문장을 자기 말로 완성하세요.

1. **관찰:** 어느 업종에서 위험 신호가 많거나 승인율이 낮은지 수치로 설명하세요.
2. **근거:** 비교한 두 집단의 실제 숫자를 적으세요.
3. **한계:** 이 결과만으로 원인을 확정할 수 없는 이유를 적으세요.

답안 예시 형식:

> A 집단의 평균 ○○은 12.4로 B 집단의 8.1보다 높았습니다. 따라서 A 집단을 먼저 확인할 수 있습니다. 다만 이 차이가 집단 자체 때문에 생겼다고 단정할 수는 없습니다.

## Q7. 터미널 두 개에서 API와 Streamlit을 차례로 실행합니다

먼저 이 노트북의 Q1~Q5를 실행해 `outputs/finance_dashboard_ready.csv`을 만듭니다.

**터미널 1: FastAPI 서버**

```bash
cd "option2_finance"
python -m uvicorn api_app_student:app --reload --host 127.0.0.1 --port 8010
```

**터미널 2: Streamlit 화면**

```bash
cd "option2_finance"
python -m streamlit run dashboard_student.py --server.port 8510
```

터미널 1은 API 요청을 기다리며 계속 실행 중이어야 합니다. 그래서 같은 터미널에 Streamlit 명령을 이어서 입력할 수 없고 터미널 2가 필요합니다.

최종 확인:

- `http://127.0.0.1:8010/health`
- `http://127.0.0.1:8010/items?limit=3`
- `http://127.0.0.1:8510`